## Pull Git Changes

Clones into `/kaggle/working/project` so `import src...` works after `sys.path` is set.


In [1]:
REPO_URL = "https://github.com/daniahsn/hotel-data-analysis.git" 

import shutil
from pathlib import Path

dest = Path("/kaggle/working/project")
if dest.exists():
    shutil.rmtree(dest)
!git clone {REPO_URL} {dest}

Cloning into '/kaggle/working/project'...
remote: Enumerating objects: 122, done.
remote: Counting objects: 100% (122/122), done.
remote: Compressing objects: 100% (77/77), done.
remote: Total 122 (delta 44), reused 106 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (122/122), 52.35 KiB | 4.76 MiB/s, done.
Resolving deltas: 100% (44/44), done.


## Pre-requisites

In the Kaggle notebook sidebar, use **Add data** and attach:

- Hotels CSV  
- World cities (`worldcitiespop.csv` or equivalent)


## Verify Input CSV exists

If anything is missing, re-check **Add data** and dataset names under `/kaggle/input`.

In [6]:
import os
from pathlib import Path

HOTELS = "/kaggle/input/datasets/raj713335/tbo-hotels-dataset/hotels.csv"
WORLD = "/kaggle/input/datasets/organizations/max-mind/world-cities-database/worldcitiespop.csv"
for name, p in [("hotels", HOTELS), ("world", WORLD)]:
    print(name, "exists:", os.path.isfile(p), p)

import sys
sys.path.insert(0, "/kaggle/working/project")

hotels exists: True /kaggle/input/datasets/raj713335/tbo-hotels-dataset/hotels.csv
world exists: True /kaggle/input/datasets/organizations/max-mind/world-cities-database/worldcitiespop.csv


In [2]:
%cd /kaggle/working/project
!pip install -q -r requirements.txt

/kaggle/working/project
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.5/47.5 kB 2.3 MB/s eta 0:00:00


## Data Wrangling

Reads the **raw Hotels CSV** and **raw World Cities CSV**

- **Standardization**
  - country → ISO2 (`country_iso2`)
  - city name normalization (`city_join_key`)
  - rating parsing (`hotel_star_rating`)
  - map parsing (`hotel_latitude`, `hotel_longitude`)
- **Feature creation**
  - `attractions_count`: number of nearby attractions mentioned in the HotelFacilities
  - `facilities_count`: number of facility/amenity tokens listed in HotelFacilities
  - `facilities_keyword_hits`: number of matched amenity keywords/phrases found in HotelFacilities
- **Join**
  - left-join hotels to world cities on `(country_iso2, city_join_key)`
  - adds `city_population`

**Outputs written**

It writes Parquet outputs to the output directory `/kaggle/working/processed`. After it finishes, you can **download** that Parquet or publish it as a **Kaggle Dataset**, then **Add data** in your modeling notebook so you can train models without rerunning the long cleaning job. 


In [6]:

!python3 scripts/pipeline/run_cleaning.py --output-dir /kaggle/working/hotel_cleaned --format parquet --chunksize 50000 --progress-every 50000

/kaggle/working/project
[2026-04-13T06:15:45] cleaned 50,000 hotel rows
[2026-04-13T06:16:15] cleaned 100,000 hotel rows
[2026-04-13T06:16:45] cleaned 150,000 hotel rows
[2026-04-13T06:17:14] cleaned 200,000 hotel rows
[2026-04-13T06:17:43] cleaned 250,000 hotel rows
[2026-04-13T06:18:13] cleaned 300,000 hotel rows
[2026-04-13T06:18:43] cleaned 350,000 hotel rows
[2026-04-13T06:19:12] cleaned 400,000 hotel rows
[2026-04-13T06:19:43] cleaned 450,000 hotel rows
[2026-04-13T06:20:15] cleaned 500,000 hotel rows
[2026-04-13T06:20:46] cleaned 550,000 hotel rows
[2026-04-13T06:21:18] cleaned 600,000 hotel rows
[2026-04-13T06:21:49] cleaned 650,000 hotel rows
[2026-04-13T06:22:19] cleaned 700,000 hotel rows
[2026-04-13T06:22:49] cleaned 750,000 hotel rows
[2026-04-13T06:23:19] cleaned 800,000 hotel rows
[2026-04-13T06:23:47] cleaned 850,000 hotel rows
[2026-04-13T06:24:18] cleaned 900,000 hotel rows
[2026-04-13T06:24:50] cleaned 950,000 hotel rows
[2026-04-13T06:25:23] cleaned 1,000,000 hotel 

## Train Baseline Regressor
- Training looks for **`hotels_with_cities*.parquet`** under `/kaggle/input` (Kaggle) or `data/processed/` (local).
-  Runs the feature pipeline on the full joined table and trains Linear Regression and Random Forest
- Prints RMSE/R² and saves each model and metrics under `/kaggle/working/model_artifacts` and `/kaggle/working/model_artifacts_rf`

In [6]:
!python3 scripts/modeling/train_baseline_model.py --model linear --project-root /kaggle/working/project

joined table: /kaggle/input/datasets/daniahasan3/hotels-with-cities-2-parquet/hotels_with_cities-2.parquet
rows used: 694,365
RMSE: 0.7553
R2:   0.1960
wrote artifacts under /kaggle/working/model_artifacts


In [7]:
!python3 scripts/modeling/train_baseline_model.py --model rf --project-root /kaggle/working/project --out-dir /kaggle/working/model_artifacts_rf

joined table: /kaggle/input/datasets/daniahasan3/hotels-with-cities-2-parquet/hotels_with_cities-2.parquet
rows used: 694,365
RMSE: 0.6531
R2:   0.3989
wrote artifacts under /kaggle/working/model_artifacts_rf
